In [1]:
import os
import json
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets, Value
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from openai import OpenAI
import modal
from litellm import completion
from tqdm.notebook import tqdm
from IPython.display import display, Markdown
from pricer.evaluator import evaluate

In [2]:
load_dotenv(override=True)

DB = "stocks-vectorstore"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [3]:
openai = OpenAI(api_key=OPENAI_API_KEY)

In [4]:
ds = load_dataset("matthewyn/stocks")
train = ds["train"]
val = ds["validation"]
test = ds["test"]

In [5]:
print(f"The length of the training set is {len(train)}, the length of the validation set is {len(val)}, and the length of the test set is {len(test)}.")

The length of the training set is 10668, the length of the validation set is 1334, and the length of the test set is 1334.


In [6]:
chroma = chromadb.PersistentClient(path=DB)

In [7]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
collection_name = "stocks"
existing_collections = [col.name for col in chroma.list_collections()]

if collection_name not in existing_collections:
    collection = chroma.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item["Price History"] for item in train.select(range(i, min(i + 1000, len(train))))]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"return_percent": item["Return %"]} for item in train.select(range(i, min(i + 1000, len(train))))]
        ids = [f"doc_{j}" for j in range(i, min(i + 1000, len(train)))]
        collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=vectors)

collection = chroma.get_or_create_collection(collection_name)

In [16]:
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "stocks"
embedding_dim = encoder.get_sentence_embedding_dimension()

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=embedding_dim,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

for i in tqdm(range(0, len(train), 1000)):
    batch = train.select(range(i, min(i + 1000, len(train))))
    
    documents = [item["Price History"] for item in batch]
    vectors = encoder.encode(documents).astype(float).tolist()
    metadatas = [
        {
            "last_price": item["Last Price"],
            "return_percent": item["Return %"],
            "document": item["Price History"]
        }
        for item in batch
    ]
    ids = [f"doc_{j}" for j in range(i, min(i + 1000, len(train)))]

    upsert_data = list(zip(ids, vectors, metadatas))
    index.upsert(vectors=upsert_data)

/var/folders/0w/xygf2w9n33xgwbk7tv9l44gw0000gn/T/ipykernel_83551/2508007024.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = encoder.get_sentence_embedding_dimension()


  0%|          | 0/11 [00:00<?, ?it/s]

In [9]:
def vector(item):
    return encoder.encode(item["Price History"])

In [10]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=3)
    documents = results["documents"][0][:]
    prices = [metadata["return_percent"] for metadata in results["metadatas"][0][:]]
    return documents, prices

In [47]:
def find_similars(item):
    vec = vector(item).astype(float).tolist()
    
    results = index.query(vector=vec, top_k=3, include_metadata=True)
    
    documents = [match["metadata"]["document"] for match in results["matches"]]
    prices = [
        (float(match["metadata"]["last_price"]), float(match["metadata"]["return_percent"]))
        for match in results["matches"]
    ]
    
    return documents, prices

In [11]:
find_similars(test[0])

(['TREND: Bearish — the stock remains below all major moving averages, indicating sustained downward pressure.\n\nMOMENTUM: Momentum is fading with recent negative readings and RSI near oversold levels, suggesting waning selling interest but no immediate reversal.\n\nMA_STRUCTURE: The price is below the MA-50, MA-150, and MA-200, with these averages diverging further from the current level, reinforcing the bearish stance.\n\nVOLUME: Volume spikes during recent declines suggest strong selling conviction, while low volume during minor rebounds indicates weak buying support.\n\nKEY_LEVELS: Support around 1625, resistance near 1780.\n\nRISK_FACTORS: The sharp decline and oversold RSI introduce risk of a further decline or short-term bounce, but the overall structure remains bearish.\n\nPREDICTION_CONTEXT: The prevailing technical signals favor continued downward momentum over the next 30 days.',
  'TREND: Bearish — the stock has been in a sustained decline well below all major moving avera

In [12]:
def make_context(similars, prices):
    context = "For context, here are some similar market summaries and their corresponding return percentages:\n\n"
    for (similar, return_percent) in zip(similars, prices):
        context += f"Market summary: \n{similar}\nReturn % (in 30 days): {return_percent}\n\n"
    return context

In [13]:
document, prices = find_similars(test[0])
print(make_context(document, prices))

For context, here are some similar market summaries and their corresponding return percentages:

Market summary: 
TREND: Bearish — the stock remains below all major moving averages, indicating sustained downward pressure.

MOMENTUM: Momentum is fading with recent negative readings and RSI near oversold levels, suggesting waning selling interest but no immediate reversal.

MA_STRUCTURE: The price is below the MA-50, MA-150, and MA-200, with these averages diverging further from the current level, reinforcing the bearish stance.

VOLUME: Volume spikes during recent declines suggest strong selling conviction, while low volume during minor rebounds indicates weak buying support.

KEY_LEVELS: Support around 1625, resistance near 1780.

RISK_FACTORS: The sharp decline and oversold RSI introduce risk of a further decline or short-term bounce, but the overall structure remains bearish.

PREDICTION_CONTEXT: The prevailing technical signals favor continued downward momentum over the next 30 days

In [14]:
def messages_for(item, similars, prices):
    message = f"Given this market summary where the last price was {item['Last Price']:.2f}, predict the percentage change in price after 30 days. Return a single number representing the percentage change (e.g. 5.2 for +5.2%, -3.1 for -3.1%); No explanation, no % symbol, no extra text. Market summary:\n\n{item['Price History']}"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [15]:
def gpt_4__1_nano_rag(item):
    documents, prices = find_similars(item)
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item, documents, prices), seed=42)
    pct_change = float(response.choices[0].message.content.strip())
    return item['Last Price'] * (1 + pct_change / 100)

In [16]:
evaluate(gpt_4__1_nano_rag, test)

  0%|          | 0/200 [00:00<?, ?it/s]

1 0 0 5 8 5 12 6 6 2 2 3 10 3 0 2 0 3 3 1 0 2 6 2 1 1 1 2 1 0 0 2 0 4 8 21 26 21 17 14 10 21 19 20 23 20 20 21 20 19 33 31 28 23 22 25 23 22 24 24 21 23 28 26 16 25 25 48 66 61 54 32 31 22 11 12 7 3 2 7 24 27 22 5 12 11 7 9 10 11 11 14 21 21 12 36 47 74 74 64 53 44 40 27 25 18 12 7 0 9 7 20 11 5 7 2 5 3 3 2 2 4 2 3 5 2 10 0 2 3 2 9 12 8 16 9 4 1 2 1 5 3 10 9 4 2 4 2 6 0 0 0 1 1 2 3 7 2 1 0 1 2 1 7 2 3 3 4 2 5 11 6 1 4 4 3 3 0 2 1 9 8 1 4 4 1 3 3 4 1 0 2 1 3 5 5 4 3 2 5 

In [23]:
!modal deploy pricer_service.py

⠴ Creating objects.....
└── ⠋ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service.py: Uploaded 
⠏ Creating objects...
└── ⠸ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service.py: Uploaded 
⠹ Creating objects...
└── ⠦ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service.py: Uploaded 
⠴ Creating objects...
└── ⠏ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service.py: Uploaded 
⠇ Creating objects...
└── ⠹ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service.py: Uploaded 
⠙ Creating objects...
└── ⠦ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service.py: Uploaded 
⠴ Creating objects...
└── ⠏ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service.py: Uploaded 
⠇ Creating objects...
└── ⠹ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service.py: Finalizing 
⠙ Creating o

In [24]:
!modal deploy pricer_service2.py

⠦ Creating objects.....
└── ⠋ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service2.py: Uploaded 
⠏ Creating objects...
└── ⠸ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service2.py: Uploaded 
⠹ Creating objects...
└── ⠦ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service2.py: Uploaded 
⠴ Creating objects...
└── ⠏ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service2.py: Uploaded 
⠏ Creating objects...
└── ⠹ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service2.py: Finalizing
⠹ Creating objects...
└── ⠴ Creating mount 
    /Users/kennethmatthewyonathan/Projects/finora/pricer_service2.py: Finalizing
⠴ Creating objects...
├── 🔨 Created mount 
│   /Users/kennethmatthewyonathan/Projects/finora/pricer_service2.py
⠇ Creating objects....
├── 🔨 Created mount 
│   /Users/kennethmatthewyonathan/Projects/finora/pricer_service2.py
⠋ Creating objects...
├── 🔨

In [ ]:
Pricer = modal.Cls.from_name("plain-pricer-service", "Stock_Pricer")
pricer = Pricer()

In [ ]:
def llama_pricer(item):
    pct_change = pricer.price.remote(item["Price History"], item["Last Price"])
    return item['Last Price'] * (1 + pct_change / 100)

In [27]:
evaluate(llama_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

2 2 2 1 2 1 1 2 1 2 2 2 1 2 1 2 2 1 6 2 1 0 1 1 2 2 1 1 1 1 1 1 1 4 8 23 27 22 18 14 10 22 19 21 24 21 20 22 21 18 34 32 29 24 23 26 24 23 25 25 22 22 29 25 4 22 24 39 49 60 32 25 18 12 6 0 5 10 14 18 21 25 18 18 16 15 16 16 13 16 17 23 20 19 18 16 49 71 71 61 61 52 22 34 22 20 20 3 21 16 8 17 8 13 2 16 0 3 4 3 6 6 21 18 19 21 22 3 21 2 20 8 1 11 16 10 3 2 1 18 4 16 18 17 0 19 2 1 8 2 1 1 0 0 3 6 6 0 0 5 3 2 2 21 18 20 2 2 1 23 23 22 3 18 4 5 22 3 3 3 6 6 4 4 4 5 4 4 4 5 3 6 0 2 24 5 10 3 2 4 

## Now, for the Llama Fine Tuning + RAG model

In [17]:
Pricer = modal.Cls.from_name("pricer-service", "Stock_Pricer")
pricer = Pricer()

In [18]:
def llama_pricer(item):
    pct_change = pricer.price.remote(item["Price History"], item["Last Price"])
    return item['Last Price'] * (1 + float(pct_change) / 100)

In [19]:
evaluate(llama_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

11 0 1 12 13 2 12 2 12 12 12 12 12 12 12 2 12 11 11 1 2 2 2 2 2 12 2 2 12 2 2 2 2 5 18 24 28 23 19 15 21 23 20 22 25 22 21 23 22 21 35 33 30 25 24 27 25 24 26 26 23 25 30 28 22 34 32 51 65 60 56 37 33 25 17 13 9 2 14 5 20 22 4 11 4 9 5 4 7 5 9 12 14 14 1 34 40 54 63 56 48 42 37 11 7 17 4 9 5 13 4 14 25 4 16 1 12 11 3 1 2 4 1 16 5 3 0 0 0 15 15 17 17 20 8 18 16 3 4 4 7 13 7 12 14 1 4 15 5 3 2 2 1 13 0 2 3 3 3 2 1 1 1 3 0 1 1 2 0 3 7 4 1 11 2 2 2 2 0 1 2 2 2 13 2 14 0 11 2 12 12 11 0 1 4 3 3 1 1 14 